# 📊 Dataset Visualization

This notebook visualizes a LeRobot dataset by plotting **observation states** and **actions** together.

**What it does:**
1. Loads a dataset
2. Extracts state and action data for each episode
3. Plots time series of states and actions
4. Provides statistics and distribution analysis

Use this to:
- Verify data quality after conversion
- Understand the relationship between states and actions
- Identify anomalies or issues in recordings

---
## 1. Configuration

> **Action Required:** Update `DATASET_DIR` below.

In [ ]:
import pathlib

# --- Dataset Path ---
# TODO: Path to the dataset to visualize
DATASET_DIR = pathlib.Path("/data/lerobot/your_dataset_here")

# --- Visualization Settings ---
# Maximum number of episodes to plot (set to None for all)
MAX_EPISODES = 5

# Which episodes to plot (None = first MAX_EPISODES, or specify list like [0, 2, 5])
EPISODE_INDICES = None

print(f"Dataset: {DATASET_DIR}")
print(f"Max episodes: {MAX_EPISODES if MAX_EPISODES else 'All'}")

---
## 2. Load Dataset

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print(f"Loading dataset from {DATASET_DIR}...")
dataset = LeRobotDataset(
    repo_id=str(DATASET_DIR),
    root=DATASET_DIR,
    video_backend="pyav",
)

print(f"\n✅ Dataset loaded!")
print(f"   Episodes: {dataset.meta.total_episodes}")
print(f"   Frames: {dataset.meta.total_frames}")
print(f"   FPS: {dataset.meta.fps}")

---
## 3. Inspect Dataset Features

Let's see what features are available in the dataset.

In [ ]:
import json

# Get feature info
features = dataset.meta.info.get("features", {})

print("Dataset Features:")
print("=" * 60)

state_names = None
action_names = None

for feature_name, feature_info in features.items():
    shape = feature_info.get("shape", "unknown")
    dtype = feature_info.get("dtype", "unknown")
    names = feature_info.get("names", [])
    
    print(f"\n{feature_name}:")
    print(f"  Shape: {shape}")
    print(f"  Dtype: {dtype}")
    if names:
        print(f"  Names: {names}")
        
    # Store names for later
    if feature_name == "observation.state":
        state_names = names
    elif feature_name == "action":
        action_names = names

print("\n" + "=" * 60)

---
## 4. Extract Episode Data

In [ ]:
from tqdm.auto import tqdm
import numpy as np

# Create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=4,
    batch_size=1,
    shuffle=False,
    drop_last=False,
)

# Determine which episodes to process
if EPISODE_INDICES is not None:
    target_episodes = set(EPISODE_INDICES)
else:
    target_episodes = None  # Will be determined by MAX_EPISODES

# Dictionary to store data for each episode
episodes_data = {}

print(f"Extracting data from episodes...")

for batch in tqdm(dataloader, desc="Loading"):
    # Get episode index
    ep_idx = int(batch["episode_index"].view(-1)[0].item())
    
    # Check if we should process this episode
    if target_episodes is not None:
        if ep_idx not in target_episodes:
            continue
    elif MAX_EPISODES is not None and ep_idx >= MAX_EPISODES:
        break
    
    # Initialize episode data structure
    if ep_idx not in episodes_data:
        episodes_data[ep_idx] = {
            "states": [],
            "actions": [],
            "times": [],
        }
    
    # Extract data
    state = batch["observation.state"].detach().float().view(-1).numpy()
    action = batch["action"].detach().float().view(-1).numpy()
    timestamp = float(batch["timestamp"].view(-1)[0].item())
    
    episodes_data[ep_idx]["states"].append(state)
    episodes_data[ep_idx]["actions"].append(action)
    episodes_data[ep_idx]["times"].append(timestamp)

# Convert to numpy arrays
for ep_idx in episodes_data:
    episodes_data[ep_idx]["states"] = np.array(episodes_data[ep_idx]["states"])
    episodes_data[ep_idx]["actions"] = np.array(episodes_data[ep_idx]["actions"])
    episodes_data[ep_idx]["times"] = np.array(episodes_data[ep_idx]["times"])

num_episodes = len(episodes_data)
print(f"\n✅ Loaded {num_episodes} episodes")

# Show episode summaries
for ep_idx in sorted(episodes_data.keys()):
    ep_data = episodes_data[ep_idx]
    duration = ep_data["times"][-1] - ep_data["times"][0] if len(ep_data["times"]) > 1 else 0
    print(f"  Episode {ep_idx}: {len(ep_data['times'])} frames, {duration:.2f}s")

---
## 5. Plot States and Actions Together

Visualize the time series of states and actions for each episode.

In [ ]:
import matplotlib.pyplot as plt

# Get dimensions
first_ep = list(episodes_data.keys())[0]
state_dim = episodes_data[first_ep]["states"].shape[1]
action_dim = episodes_data[first_ep]["actions"].shape[1]

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"\nPlotting {num_episodes} episodes...")

In [ ]:
# Plot each episode separately
for ep_idx in sorted(episodes_data.keys()):
    ep_data = episodes_data[ep_idx]
    times = ep_data["times"]
    states = ep_data["states"]
    actions = ep_data["actions"]
    
    # Create figure with two subplots: states and actions
    fig, (ax_state, ax_action) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Plot states
    for d in range(state_dim):
        label = state_names[d] if state_names and d < len(state_names) else f"state_{d}"
        ax_state.plot(times, states[:, d], alpha=0.7, label=label)
    
    ax_state.set_ylabel("State Value")
    ax_state.set_title(f"Episode {ep_idx}: Observation States")
    ax_state.grid(True, linestyle="--", alpha=0.3)
    ax_state.legend(loc="upper right", fontsize='x-small', ncol=2)
    
    # Plot actions
    for d in range(action_dim):
        label = action_names[d] if action_names and d < len(action_names) else f"action_{d}"
        ax_action.plot(times, actions[:, d], alpha=0.7, label=label)
    
    ax_action.set_xlabel("Time (s)")
    ax_action.set_ylabel("Action Value")
    ax_action.set_title(f"Episode {ep_idx}: Actions")
    ax_action.grid(True, linestyle="--", alpha=0.3)
    ax_action.legend(loc="upper right", fontsize='x-small', ncol=2)
    
    plt.tight_layout()
    plt.show()

---
## 6. Compare Episodes (Overlaid)

Plot all episodes overlaid to compare trajectories.

In [ ]:
# Select a subset of dimensions to plot (for readability)
# Modify these to focus on specific features
STATE_DIMS_TO_PLOT = list(range(min(6, state_dim)))  # First 6 state dims
ACTION_DIMS_TO_PLOT = list(range(min(6, action_dim)))  # First 6 action dims

fig, axes = plt.subplots(len(STATE_DIMS_TO_PLOT) + len(ACTION_DIMS_TO_PLOT), 1, 
                          figsize=(14, 2 * (len(STATE_DIMS_TO_PLOT) + len(ACTION_DIMS_TO_PLOT))), 
                          sharex=True)

# Plot states
for i, d in enumerate(STATE_DIMS_TO_PLOT):
    ax = axes[i]
    for ep_idx in sorted(episodes_data.keys()):
        ep_data = episodes_data[ep_idx]
        color = f"C{ep_idx % 10}"
        label = f"Ep {ep_idx}" if i == 0 else None
        ax.plot(ep_data["times"], ep_data["states"][:, d], 
                color=color, alpha=0.7, label=label)
    
    name = state_names[d] if state_names and d < len(state_names) else f"state_{d}"
    ax.set_ylabel(name, fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.3)
    if i == 0:
        ax.set_title("Observation States (All Episodes Overlaid)")
        ax.legend(loc="upper right", fontsize='x-small', ncol=min(5, num_episodes))

# Plot actions
for i, d in enumerate(ACTION_DIMS_TO_PLOT):
    ax = axes[len(STATE_DIMS_TO_PLOT) + i]
    for ep_idx in sorted(episodes_data.keys()):
        ep_data = episodes_data[ep_idx]
        color = f"C{ep_idx % 10}"
        ax.plot(ep_data["times"], ep_data["actions"][:, d], 
                color=color, alpha=0.7)
    
    name = action_names[d] if action_names and d < len(action_names) else f"action_{d}"
    ax.set_ylabel(name, fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.3)
    if i == 0:
        ax.set_title("Actions (All Episodes Overlaid)")

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

---
## 7. State-Action Correlation

Analyze the relationship between states and actions.

In [ ]:
# Concatenate all episode data
all_states = np.concatenate([episodes_data[ep]["states"] for ep in sorted(episodes_data.keys())], axis=0)
all_actions = np.concatenate([episodes_data[ep]["actions"] for ep in sorted(episodes_data.keys())], axis=0)

print(f"Total samples: {len(all_states)}")
print(f"State shape: {all_states.shape}")
print(f"Action shape: {all_actions.shape}")

In [ ]:
# Compute correlation matrix between states and actions
combined = np.concatenate([all_states, all_actions], axis=1)
correlation = np.corrcoef(combined.T)

# Extract state-action correlation block
state_action_corr = correlation[:state_dim, state_dim:]

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(12, 8))

im = ax.imshow(state_action_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# Labels
state_labels = state_names if state_names else [f"s{i}" for i in range(state_dim)]
action_labels = action_names if action_names else [f"a{i}" for i in range(action_dim)]

ax.set_xticks(range(action_dim))
ax.set_xticklabels(action_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(state_dim))
ax.set_yticklabels(state_labels, fontsize=8)

ax.set_xlabel("Actions")
ax.set_ylabel("States")
ax.set_title("State-Action Correlation Matrix")

plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

---
## 8. Statistics Summary

In [ ]:
print("State Statistics:")
print("=" * 80)
print(f"{'Feature':<30} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12}")
print("-" * 80)

for d in range(state_dim):
    name = state_names[d] if state_names and d < len(state_names) else f"state_{d}"
    values = all_states[:, d]
    print(f"{name:<30} {np.mean(values):>12.4f} {np.std(values):>12.4f} "
          f"{np.min(values):>12.4f} {np.max(values):>12.4f}")

print("\n")
print("Action Statistics:")
print("=" * 80)
print(f"{'Feature':<30} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12}")
print("-" * 80)

for d in range(action_dim):
    name = action_names[d] if action_names and d < len(action_names) else f"action_{d}"
    values = all_actions[:, d]
    print(f"{name:<30} {np.mean(values):>12.4f} {np.std(values):>12.4f} "
          f"{np.min(values):>12.4f} {np.max(values):>12.4f}")

---
## 9. Distribution Plots

In [ ]:
# Action distribution histograms
n_cols = min(4, action_dim)
n_rows = (action_dim + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = np.atleast_2d(axes)

for d in range(action_dim):
    row, col = d // n_cols, d % n_cols
    ax = axes[row, col]
    
    name = action_names[d] if action_names and d < len(action_names) else f"action_{d}"
    ax.hist(all_actions[:, d], bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel(name)
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)

# Hide empty subplots
for d in range(action_dim, n_rows * n_cols):
    row, col = d // n_cols, d % n_cols
    axes[row, col].set_visible(False)

plt.suptitle('Action Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# State distribution histograms (if not too many dimensions)
if state_dim <= 20:
    n_cols = min(4, state_dim)
    n_rows = (state_dim + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
    axes = np.atleast_2d(axes)
    
    for d in range(state_dim):
        row, col = d // n_cols, d % n_cols
        ax = axes[row, col]
        
        name = state_names[d] if state_names and d < len(state_names) else f"state_{d}"
        ax.hist(all_states[:, d], bins=50, edgecolor='black', alpha=0.7, color='green')
        ax.set_xlabel(name)
        ax.set_ylabel('Count')
        ax.grid(True, alpha=0.3)
    
    # Hide empty subplots
    for d in range(state_dim, n_rows * n_cols):
        row, col = d // n_cols, d % n_cols
        axes[row, col].set_visible(False)
    
    plt.suptitle('State Distributions', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"Skipping state distribution plots ({state_dim} dimensions is too many)")

---
## ✅ Done!

**What to look for:**

- **Consistent trajectories**: Similar patterns across episodes indicate good data quality
- **Smooth transitions**: Jerky or noisy data may indicate recording issues
- **Reasonable ranges**: Check that values are within expected physical limits
- **Correlations**: High state-action correlations suggest the policy should be learnable
- **Outliers**: Unusual spikes may indicate sensor errors or edge cases

**Common issues:**
- Large gaps in time: May need to filter or re-record
- Constant values: Sensor may not be updating
- Extreme outliers: May need to clip or remove bad frames